In [1]:
"""
MAP Competition - Simple DeepSeek-7B Inference
Single model inference for Kaggle submission
"""

import os
import gc
import warnings
warnings.filterwarnings('ignore')

def log(msg):
    print(msg, flush=True)

log("Starting imports...")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from scipy.special import softmax
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_PATH = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
TEST_PATH = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
MODEL_PATH = "/kaggle/input/datasets/mdmarufhasanrubab/map-ds-deepseek7b"
MAX_LEN = 256
BATCH_SIZE = 4

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_model_path(base_path):
    """Find the actual model directory within a dataset."""
    import glob

    if not os.path.exists(base_path):
        return base_path

    # Check base path first
    if os.path.exists(os.path.join(base_path, "config.json")):
        return base_path

    # Search for config.json with model weights
    for depth in range(1, 5):
        pattern = os.path.join(base_path, "/".join(["*"] * depth), "config.json")
        matches = glob.glob(pattern)
        for m in matches:
            dir_path = os.path.dirname(m)
            has_weights = (os.path.exists(os.path.join(dir_path, "model.safetensors")) or
                          os.path.exists(os.path.join(dir_path, "pytorch_model.bin")) or
                          glob.glob(os.path.join(dir_path, "model*.safetensors")))
            if has_weights:
                return dir_path

    return base_path


def format_input(row):
    """Format input text for DeepSeek model."""
    x = "This answer is correct." if row['is_correct'] else "This answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


# ============================================================================
# MAIN
# ============================================================================

def main():
    log("="*60)
    log("MAP COMPETITION - DEEPSEEK-7B INFERENCE")
    log("="*60)

    if torch.cuda.is_available():
        log(f"GPU: {torch.cuda.get_device_name(0)}")

    # Load data
    log("Loading data...")
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    log(f"Train: {train.shape}, Test: {test.shape}")

    # Prepare labels
    train['Misconception'] = train['Misconception'].fillna('NA')
    train['target'] = train['Category'] + ':' + train['Misconception']

    le = LabelEncoder()
    le.fit(train['target'])
    n_classes = len(le.classes_)
    log(f"Number of classes: {n_classes}")

    # Find correct answers for is_correct flag
    idx = train['Category'].apply(lambda c: c.split('_')[0]) == 'True'
    correct = train.loc[idx].copy()
    correct['c'] = correct.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
    correct = correct.sort_values('c', ascending=False).drop_duplicates(['QuestionId'])
    correct = correct[['QuestionId', 'MC_Answer']]
    correct['is_correct'] = 1

    # Merge is_correct into test
    test = test.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
    test['is_correct'] = test['is_correct'].fillna(0)

    # Format input text
    test['text'] = test.apply(format_input, axis=1)

    # Build family map for filtering
    fam_map = {}
    for _, row in test.iterrows():
        fam_map[row['row_id']] = 'True_' if row['is_correct'] == 1 else 'False_'

    # Find and load model
    log(f"Finding model in: {MODEL_PATH}")
    model_path = find_model_path(MODEL_PATH)
    log(f"Found model at: {model_path}")

    # Load tokenizer
    log("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model
    log("Loading model...")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=n_classes,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map=device
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    log("Model loaded!")

    # Tokenize
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    ds_test = Dataset.from_pandas(test[['text']])
    ds_test = ds_test.map(tokenize, batched=True, remove_columns=['text'])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    dataloader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

    # Run inference
    log(f"Running inference on {len(ds_test)} samples...")
    all_logits = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inference"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.float().cpu().numpy())

    predictions = np.concatenate(all_logits, axis=0)
    probs = softmax(predictions, axis=1)
    top_indices = np.argsort(-probs, axis=1)

    # Decode labels
    flat_indices = top_indices.flatten()
    decoded_labels = le.inverse_transform(flat_indices)
    top_labels = decoded_labels.reshape(top_indices.shape)

    # Apply family filter and create submission
    log("Creating submission with family filtering...")
    final_predictions = []

    for i, row_id in enumerate(test.row_id.values):
        pref = fam_map.get(row_id, 'False_')

        # Filter predictions by family prefix
        filtered = [lbl for lbl in top_labels[i] if lbl.startswith(pref)]

        # Take top 3
        top3 = filtered[:3]

        # Fill if needed
        while len(top3) < 3:
            filler = f"{pref}Neither:NA"
            if filler not in top3:
                top3.append(filler)
            else:
                top3.append(f"{pref}Correct:NA" if pref == "True_" else f"{pref}Misconception:NA")

        final_predictions.append(" ".join(top3[:3]))

    # Save submission
    submission = pd.DataFrame({
        'row_id': test.row_id.values,
        'Category:Misconception': final_predictions
    })
    submission.to_csv('submission.csv', index=False)

    log("\n" + "="*60)
    log("SUBMISSION SAVED: submission.csv")
    log("="*60)
    log(str(submission.head(10)))
    log(f"\nTotal predictions: {len(submission)}")

    # Cleanup
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

Starting imports...
MAP COMPETITION - DEEPSEEK-7B INFERENCE
GPU: Tesla T4
Loading data...
Train: (36696, 7), Test: (3, 5)
Number of classes: 65
Finding model in: /kaggle/input/datasets/mdmarufhasanrubab/map-ds-deepseek7b
Found model at: /kaggle/input/datasets/mdmarufhasanrubab/map-ds-deepseek7b/saved_model/deepseek_model
Loading tokenizer...
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

Model loaded!


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Running inference on 3 samples...


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]

Creating submission with family filtering...

SUBMISSION SAVED: submission.csv
   row_id                             Category:Misconception
0   36696  True_Correct:NA True_Neither:NA True_Misconcep...
1   36697  False_Misconception:WNB False_Neither:NA False...
2   36698  True_Neither:NA True_Correct:NA True_Misconcep...

Total predictions: 3
